# 线性回归从零开始实现
我们将从零开始实现整个方法,包括数据流水线 模型 损失函数和小批量随机梯度下降优化器

In [ ]:

# 将matplotlib的图嵌入到代码中
%matplotlib inline
# 用于生成随机下降梯度
import random
import torch
from d2l import torch as d2l


# 一个简化模型
- 假设1:影响房价的关键因素是卧室个数,卫生间个数和居住面积,记为x1,x2,x3
- 假设2:成交价是关键因素的加权和
    y=w1x1+w2x2+w3x3+b
权重和偏差的实际值在后面决定

# 线性模型
- 给定n维输入 X=[x1,x2,...,xn]^T
- 线性模型有一个n维权重和一个标量偏差
    W = [w1,w2,...,wn]^T,b
- 输出是输入的加权和
    y=w1x1+w2x2+...+wnxn+b
向量版本:y={W,X}+b

# 衡量预估质量
- 比较真实值和预估值,例如房屋售价和估值
- 假设y是真实值,y_hat是估计值,可以比较
    l(y,y_hat)=1/2*(y-y_hat)^2
这个叫做平方损失

# 训练数据
- 收集一些数据点来决定参数值(权重和偏差),例如过去6个月卖的房子
- 这被称之为训练数据
- 通常越多越好

# 参数学习
- 计算训练损失
- 最小化损失来学习参数(使用梯度来计算出权重和偏差)

# 显式解
- 将偏差加入权重
- 损失是凸函数,所以最优解满足导函数极值最小



In [ ]:
# 生成数据
def synthetic_data(w,b,num_examples):
    """生成 y=Xw+b+噪声"""
    # 是一个均值为0，方差为1的随机数，生成num_examples个样本，列数为w的长度
    X = torch.normal(0,1,(num_examples,len(w)))
    # y 的公式
    y = torch.matmul(X,w)+b
    # y 再加上均值为0，方差为0.01，形状与y的长度一样的随机噪声
    y += torch.normal(0,0.01,y.shape)
    # 最后做成一个列向量返回
    return X,y.reshape((-1,1))

true_w = torch.tensor([2,-3.4])
true_b = 4.2
features,labels = synthetic_data(true_w,true_b,1000)
print('features:',features[0],"\nlabels:",labels[0])

In [ ]:
d2l.set_figsize()
# 对图像进行绘制
d2l.plt.scatter(features[:,1].detach().numpy(),labels.detach().numpy(),1)

In [ ]:
# 定义一个data_iter函数,该函数接受批量大小 特征矩阵和标签向量作为输入,生成大小为batch_size小批量
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    # 生成每个样本的index
    indices = list(range(num_examples))
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        # i+batch_size可能超出样本个数,最后一个批量如果没有拉满,则返回最后到样本个数的数量
        batch_indices = torch.tensor(indices[i:min(i + batch_size,num_examples)])
        # yield生成器语法:这一批数据返回除去,但是函数没有返回而是暂停在这,下次执行继续从这跑
        # 这种方式不会把数据一次性加载到内存,而是训练到哪加载到哪
        yield features[batch_indices], labels[batch_indices]


batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break

In [ ]:
# 定义模型初始化参数
w = torch.normal(0,0.01,size=(2,1),requires_grad=True)
b = torch.zeros(1,requires_grad=True)

In [ ]:
# 定义模型
def linreg(X, w, b):
    """线性回归模型"""
    return torch.matmul(X,w) + b

In [ ]:
# 定义损失函数
def squared_loss(y_hat, y):
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 /2

In [ ]:
# 定义优化算法
def sgd(params,lr,batch_size):
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad/batch_size
            # 把梯度设置为0
            param.grad.zero_()


In [ ]:


# 学习率
lr = 0.03
num_epochs = 3
# 定义参数，用于更方便的切换模型
net = linreg
# 损失函数，用于更方便的切换
loss = squared_loss
# 训练过程
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        # X和y的小批量损失
        L = loss(net(X, w, b), y)
        # 因为L的形状是(batch_size,1)，而不是标量，L中所有元素被加到
        # 并以此计算关于[w,b]的梯度
        L.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = loss(net(features,w,b), labels)
        print(f'epoch: {epoch + 1}, loss: {float(train_l.mean()):.8f}')


In [ ]:
# 比较真实参数和通过训练学到的参数来评估训练的成功程度
print(f'w的估计误差:{true_w-w.reshape(true_w.shape)}')
print(f'b的估计误差:{true_b-b}')